In [ ]:
import os
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from joblib import Parallel, delayed
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
np.set_printoptions(precision=6, suppress=True)

SEED = 42
N_SPLITS = 10
MAX_SEQ_EPOCHS = 12
SEQ_PATIENCE = 3
TABULAR_PARALLEL_JOBS = max(1, min(4, os.cpu_count() or 4))
MODEL_THREAD_COUNT = 1

DATA_ROOT = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
if not DATA_ROOT.exists():
    DATA_ROOT = Path.cwd() / "competitions" / "rogii-wellbore-geology-prediction"
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)

DEBUG_MODE = True 


def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy, and PyTorch for deterministic runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"

print(f"Seed set to: {SEED}")
print(f"Device: {DEVICE}")
print(f"Train dir exists: {TRAIN_DIR.exists()} | {TRAIN_DIR}")
print(f"Test dir exists: {TEST_DIR.exists()} | {TEST_DIR}")
print(f"Artifacts dir: {ARTIFACT_DIR}")
print(f"Library check -> torch {torch.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

In [ ]:
from typing import Dict, List, Tuple


def extract_wellname(path: Path) -> str:
    """Extract WELLNAME from a file name like WELL__horizontal_well.csv."""
    name = path.name
    return name.split("__", 1)[0] if "__" in name else path.stem


def read_csv_safe(path: Path) -> pd.DataFrame:
    """Read a CSV and attach the original row order for later alignment."""
    df = pd.read_csv(path).copy()
    df["row_idx"] = np.arange(len(df), dtype=np.int64)
    return df


def _nan_run_profile(values: np.ndarray) -> Tuple[List[int], List[int]]:
    starts, lengths = [], []
    in_run = False
    run_start = None
    for idx, is_nan in enumerate(np.isnan(values)):
        if is_nan and not in_run:
            in_run = True
            run_start = idx
        elif not is_nan and in_run:
            in_run = False
            starts.append(int(run_start))
            lengths.append(int(idx - run_start))
    if in_run and run_start is not None:
        starts.append(int(run_start))
        lengths.append(int(len(values) - run_start))
    return starts, lengths


def audit_tvt_boundary(df: pd.DataFrame, source_col: str | None = None, window: int = 5) -> dict:
    """Profile the TVT/TVT_input missingness topology and extract boundary features."""
    if source_col is None:
        if "TVT_input" in df.columns:
            source_col = "TVT_input"
        elif "TVT" in df.columns:
            source_col = "TVT"
        else:
            source_col = None

    if source_col is None:
        values = np.full(len(df), np.nan, dtype=float)
    else:
        values = pd.to_numeric(df[source_col], errors="coerce").to_numpy(dtype=float, copy=True)

    valid = np.isfinite(values)
    if valid.any():
        valid_idx = np.flatnonzero(valid)
        last_valid_idx = int(valid_idx[-1])
        transitions = np.flatnonzero(valid[:-1] & ~valid[1:]) + 1 if len(values) > 1 else np.array([], dtype=int)
        eval_start_idx = int(transitions[0]) if transitions.size else int(np.flatnonzero(~valid)[0]) if (~valid).any() else len(values)
        tail_start = max(0, last_valid_idx - window + 1)
        tail_vals = values[tail_start:last_valid_idx + 1]
        x = np.arange(len(tail_vals), dtype=float)
        local_gradient = float(np.polyfit(x, tail_vals, deg=1)[0]) if len(tail_vals) >= 2 else 0.0
        if len(tail_vals) >= 3:
            curvature = float(np.gradient(np.gradient(tail_vals))[-1])
        else:
            curvature = 0.0
        last_valid_value = float(values[last_valid_idx])
        nan_starts, nan_lengths = _nan_run_profile(values)
        terminal_nan_run = bool(eval_start_idx < len(values) and np.all(np.isnan(values[eval_start_idx:])))
    else:
        last_valid_idx = -1
        eval_start_idx = len(values)
        local_gradient = 0.0
        curvature = 0.0
        last_valid_value = np.nan
        nan_starts, nan_lengths = [], []
        terminal_nan_run = False

    missing_fraction = float(np.isnan(values).mean()) if len(values) else 0.0
    valid_prefix_len = int(eval_start_idx if eval_start_idx <= len(values) else len(values))

    return {
        "tvt_source_col": source_col,
        "tvt_eval_start_idx": int(eval_start_idx),
        "tvt_last_valid_idx": int(last_valid_idx),
        "tvt_last_valid_value": last_valid_value,
        "tvt_local_gradient": float(local_gradient),
        "tvt_curvature": float(curvature),
        "tvt_missing_fraction": missing_fraction,
        "tvt_valid_prefix_len": valid_prefix_len,
        "tvt_nan_run_starts": nan_starts,
        "tvt_nan_run_lengths": nan_lengths,
        "tvt_terminal_nan_run": terminal_nan_run,
    }


def build_boundary_features(df: pd.DataFrame, boundary: dict) -> pd.DataFrame:
    """Broadcast boundary audit features to every row in a well dataframe."""
    out = df.copy()
    for key, value in boundary.items():
        if isinstance(value, (list, tuple, np.ndarray)):
            continue
        out[key] = value
    out["tvt_rows_from_boundary"] = out["row_idx"] - out["tvt_eval_start_idx"]
    out["tvt_is_eval_zone"] = (out["row_idx"] >= out["tvt_eval_start_idx"]).astype(np.int8)
    out["tvt_has_boundary"] = int(out["tvt_eval_start_idx"].iloc[0] < len(out))
    return out


def load_well_directory(base_dir: Path) -> List[dict]:
    """Load horizontal/typewell CSV pairs from a directory, grouped by WELLNAME."""
    horizontal_paths = sorted(base_dir.glob("*__horizontal_well.csv"))
    typewell_paths = sorted(base_dir.glob("*__typewell.csv"))
    horizontal_map = {extract_wellname(p): p for p in horizontal_paths}
    typewell_map = {extract_wellname(p): p for p in typewell_paths}
    well_names = sorted(set(horizontal_map) & set(typewell_map))

    bundles = []
    for well_name in well_names:
        h_path = horizontal_map[well_name]
        t_path = typewell_map[well_name]
        h_df = read_csv_safe(h_path)
        t_df = read_csv_safe(t_path)
        h_df["WELLNAME"] = well_name
        t_df["WELLNAME"] = well_name
        h_df["Horizontal_Well_GR"] = pd.to_numeric(h_df.get("GR"), errors="coerce")
        t_df["Typewell_GR"] = pd.to_numeric(t_df.get("GR"), errors="coerce")
        t_df["Typewell_TVT"] = pd.to_numeric(t_df.get("TVT"), errors="coerce") if "TVT" in t_df.columns else np.nan
        boundary = audit_tvt_boundary(h_df)
        h_df = build_boundary_features(h_df, boundary)
        bundles.append(
            {
                "WELLNAME": well_name,
                "horizontal": h_df,
                "typewell": t_df,
                "boundary": boundary,
                "horizontal_path": h_path,
                "typewell_path": t_path,
            }
        )
    return bundles


train_wells = load_well_directory(TRAIN_DIR)
test_wells = load_well_directory(TEST_DIR)
train_well_map = {w["WELLNAME"]: w for w in train_wells}
test_well_map = {w["WELLNAME"]: w for w in test_wells}

print(f"Training wells loaded: {len(train_wells)}")
print(f"Test wells loaded: {len(test_wells)}")
if train_wells:
    sample = train_wells[0]
    print(f"Sample train horizontal shape: {sample['horizontal'].shape}")
    print(f"Sample train typewell shape: {sample['typewell'].shape}")
    print(f"Boundary audit preview: {sample['boundary']}")
    print(f"Train horizontal columns: {list(sample['horizontal'].columns[:20])}")

In [ ]:
SURFACE_COLUMNS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
SURFACE_MODEL_FEATURES = ["MD", "X", "Y", "Z"]
SURFACE_FALLBACK_FEATURE_COLUMNS = [
    f"{col}_{stat}"
    for col in SURFACE_MODEL_FEATURES
    for stat in ["mean", "std", "first", "last"]
] + ["row_count", "md_span", "z_span"]
BASE_IGNORE_COLUMNS = {
    "TVT",
    "TVT_input",
    "dtvt",
    "target_dtvt",
    "WELLNAME",
    "row_idx",
    "horizontal_path",
    "typewell_path",
}


def build_surface_lookup_from_typewell(typewell_df: pd.DataFrame) -> dict:
    """Derive deterministic formation top TVT depths from a well's typewell log."""
    tw = typewell_df.copy()
    tw["TVT"] = pd.to_numeric(tw.get("TVT"), errors="coerce")
    tw["Geology"] = tw.get("Geology", pd.Series(index=tw.index, dtype=object)).astype(str).str.strip().str.upper()
    tw = tw.dropna(subset=["TVT", "Geology"]).sort_values("TVT").reset_index(drop=True)
    if tw.empty:
        return {surface: np.nan for surface in SURFACE_COLUMNS}
    segment_starts = tw["Geology"].ne(tw["Geology"].shift())
    tops = tw.loc[segment_starts, ["Geology", "TVT"]]
    label_to_tvt = {row.Geology: float(row.TVT) for row in tops.itertuples(index=False)}
    return {surface: float(label_to_tvt.get(surface, np.nan)) for surface in SURFACE_COLUMNS}


def _build_surface_fallback_features(horizontal_df: pd.DataFrame) -> dict:
    """Summarize a well's base spatial geometry for the surface fallback models."""
    numeric = horizontal_df[SURFACE_MODEL_FEATURES].apply(pd.to_numeric, errors="coerce") if set(SURFACE_MODEL_FEATURES).issubset(horizontal_df.columns) else pd.DataFrame()
    features = {}
    for col in SURFACE_MODEL_FEATURES:
        values = numeric[col].to_numpy(dtype=np.float32) if col in numeric.columns else np.array([], dtype=np.float32)
        finite = values[np.isfinite(values)]
        features[f"{col}_mean"] = float(np.nanmean(values)) if finite.size else np.nan
        features[f"{col}_std"] = float(np.nanstd(values)) if finite.size else np.nan
        features[f"{col}_first"] = float(finite[0]) if finite.size else np.nan
        features[f"{col}_last"] = float(finite[-1]) if finite.size else np.nan
    features["row_count"] = float(len(horizontal_df))
    md_values = numeric["MD"].to_numpy(dtype=np.float32) if "MD" in numeric.columns else np.array([], dtype=np.float32)
    z_values = numeric["Z"].to_numpy(dtype=np.float32) if "Z" in numeric.columns else np.array([], dtype=np.float32)
    features["md_span"] = float(np.nanmax(md_values) - np.nanmin(md_values)) if np.isfinite(md_values).any() else np.nan
    features["z_span"] = float(np.nanmax(z_values) - np.nanmin(z_values)) if np.isfinite(z_values).any() else np.nan
    return features


def attach_surface_depths(horizontal_df: pd.DataFrame, surface_lookup: dict) -> pd.DataFrame:
    """Broadcast the chosen formation tops across every row of a horizontal well."""
    out = horizontal_df.copy()
    for surface in SURFACE_COLUMNS:
        value = surface_lookup.get(surface, np.nan)
        out[surface] = float(value) if pd.notna(value) else np.nan
    return out


def augment_surface_bundles(wells: List[dict]) -> List[dict]:
    """Attach the deterministic typewell lookup as the primary surface source."""
    augmented = []
    for bundle in wells:
        updated = dict(bundle)
        updated["surface_lookup_primary"] = build_surface_lookup_from_typewell(bundle["typewell"])
        augmented.append(updated)
    return augmented


def fit_surface_fallback_models(wells: List[dict]) -> tuple[dict, dict, dict]:
    """Train one LightGBM regressor per surface using well-level spatial summaries."""
    feature_rows = []
    per_surface_rows = {surface: [] for surface in SURFACE_COLUMNS}
    for bundle in wells:
        features = _build_surface_fallback_features(bundle["horizontal"])
        feature_rows.append(features)
        for surface in SURFACE_COLUMNS:
            target = bundle["surface_lookup_primary"].get(surface, np.nan)
            if pd.notna(target):
                row = dict(features)
                row["target"] = float(target)
                per_surface_rows[surface].append(row)

    feature_medians = pd.DataFrame(feature_rows)[SURFACE_FALLBACK_FEATURE_COLUMNS].median(numeric_only=True).to_dict() if feature_rows else {col: 0.0 for col in SURFACE_FALLBACK_FEATURE_COLUMNS}
    target_medians = {}
    models = {}

    for surface, rows in per_surface_rows.items():
        if not rows:
            models[surface] = None
            target_medians[surface] = np.nan
            continue

        surface_df = pd.DataFrame(rows)
        surface_df[SURFACE_FALLBACK_FEATURE_COLUMNS] = surface_df[SURFACE_FALLBACK_FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
        surface_df[SURFACE_FALLBACK_FEATURE_COLUMNS] = surface_df[SURFACE_FALLBACK_FEATURE_COLUMNS].fillna(pd.Series(feature_medians))
        X = surface_df[SURFACE_FALLBACK_FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        y = surface_df["target"].to_numpy(dtype=np.float32)
        target_medians[surface] = float(np.nanmedian(y)) if len(y) else np.nan
        if len(surface_df) < 5:
            models[surface] = None
            continue
        model = lgb.LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            objective="regression",
            force_col_wise=True,
            verbosity=-1,
        )
        model.fit(X, y)
        models[surface] = model

    return models, feature_medians, target_medians


def fill_surface_lookup_with_fallback(bundle: dict, fallback_models: dict, feature_medians: dict, target_medians: dict) -> dict:
    """Patch any missing geological surfaces with the auxiliary LightGBM predictors."""
    lookup = dict(bundle["surface_lookup_primary"])
    feature_row = _build_surface_fallback_features(bundle["horizontal"])
    feature_row = pd.Series(feature_row).reindex(SURFACE_FALLBACK_FEATURE_COLUMNS)
    feature_row = feature_row.fillna(pd.Series(feature_medians)).astype(np.float32)
    feature_frame = pd.DataFrame([feature_row], columns=SURFACE_FALLBACK_FEATURE_COLUMNS)

    for surface in SURFACE_COLUMNS:
        if pd.notna(lookup.get(surface, np.nan)):
            continue
        model = fallback_models.get(surface)
        if model is not None:
            lookup[surface] = float(model.predict(feature_frame)[0])
        if pd.isna(lookup.get(surface, np.nan)):
            lookup[surface] = float(target_medians.get(surface, np.nan)) if pd.notna(target_medians.get(surface, np.nan)) else np.nan
    return lookup


def finalize_surface_bundles(wells: List[dict], fallback_models: dict, feature_medians: dict, target_medians: dict) -> List[dict]:
    """Materialize fully populated surface markers on each well bundle."""
    finalized = []
    for bundle in wells:
        updated = dict(bundle)
        updated["surface_lookup"] = fill_surface_lookup_with_fallback(bundle, fallback_models, feature_medians, target_medians)
        updated["horizontal"] = attach_surface_depths(bundle["horizontal"], updated["surface_lookup"])
        finalized.append(updated)
    return finalized


train_wells = augment_surface_bundles(train_wells)
test_wells = augment_surface_bundles(test_wells)
surface_fallback_models, surface_feature_medians, surface_target_medians = fit_surface_fallback_models(train_wells)
train_wells = finalize_surface_bundles(train_wells, surface_fallback_models, surface_feature_medians, surface_target_medians)
test_wells = finalize_surface_bundles(test_wells, surface_fallback_models, surface_feature_medians, surface_target_medians)
train_well_map = {w["WELLNAME"]: w for w in train_wells}
test_well_map = {w["WELLNAME"]: w for w in test_wells}

if train_wells:
    print(f"Sample primary surface lookup: {train_wells[0]['surface_lookup_primary']}")
    print(f"Sample final surface lookup: {train_wells[0]['surface_lookup']}")

train_surface_nan_counts = pd.DataFrame([bundle["horizontal"][SURFACE_COLUMNS].isna().sum() for bundle in train_wells]).sum().to_dict() if train_wells else {surface: 0 for surface in SURFACE_COLUMNS}
test_surface_nan_counts = pd.DataFrame([bundle["horizontal"][SURFACE_COLUMNS].isna().sum() for bundle in test_wells]).sum().to_dict() if test_wells else {surface: 0 for surface in SURFACE_COLUMNS}
print(f"Filled train surface matrix shape: ({sum(len(bundle['horizontal']) for bundle in train_wells)}, {len(SURFACE_COLUMNS)})")
print(f"Filled test surface matrix shape: ({sum(len(bundle['horizontal']) for bundle in test_wells)}, {len(SURFACE_COLUMNS)})")
print(f"Train surface NaN counts: {train_surface_nan_counts}")
print(f"Test surface NaN counts: {test_surface_nan_counts}")


def engineer_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create geometry, boundary, and surface-distance features for tabular models."""
    out = df.copy()
    for col in ["MD", "X", "Y", "Z", "GR", "Horizontal_Well_GR"] + SURFACE_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "Z" in out.columns:
        for surface in SURFACE_COLUMNS:
            out[f"abs_Z_minus_{surface}"] = (out["Z"] - out[surface]).abs() if surface in out.columns else np.nan

    if "GR" in out.columns:
        out["GR_roll_mean_5"] = out["GR"].rolling(5, min_periods=1).mean()
        out["GR_roll_std_5"] = out["GR"].rolling(5, min_periods=1).std().fillna(0.0)
        out["GR_diff_1"] = out["GR"].diff().fillna(0.0)
    else:
        out["GR_roll_mean_5"] = np.nan
        out["GR_roll_std_5"] = np.nan
        out["GR_diff_1"] = np.nan

    if "MD" in out.columns:
        out["MD_step"] = out["MD"].diff().fillna(0.0)
    else:
        out["MD_step"] = np.nan

    if "tvt_eval_start_idx" in out.columns:
        out["tvt_rows_from_boundary"] = out["row_idx"] - out["tvt_eval_start_idx"]
        out["tvt_is_eval_zone"] = (out["row_idx"] >= out["tvt_eval_start_idx"]).astype(np.int8)
        out["tvt_boundary_distance"] = out["row_idx"] - out["tvt_eval_start_idx"]
    else:
        out["tvt_rows_from_boundary"] = np.nan
        out["tvt_is_eval_zone"] = 0
        out["tvt_boundary_distance"] = np.nan

    return out


train_feature_frames = [engineer_tabular_features(bundle["horizontal"]) for bundle in train_wells]
test_feature_frames = [engineer_tabular_features(bundle["horizontal"]) for bundle in test_wells]

train_tabular_df = pd.concat(train_feature_frames, ignore_index=True)
test_tabular_df = pd.concat(test_feature_frames, ignore_index=True)

train_tabular_df["dtvt"] = train_tabular_df.groupby("WELLNAME")["TVT"].diff().fillna(0.0)
train_tabular_df["dtvt"] = pd.to_numeric(train_tabular_df["dtvt"], errors="coerce").fillna(0.0)

if DEBUG_MODE:
    import random
    random.seed(42)
    unique_wells = train_tabular_df["WELLNAME"].unique().tolist()
    sample_wells = random.sample(list(unique_wells), k=min(773, len(unique_wells)))
    train_tabular_df = train_tabular_df[train_tabular_df["WELLNAME"].isin(sample_wells)].reset_index(drop=True)
    print(f"DEBUGGING MODE: Downsampled training matrix to {train_tabular_df.shape}")

train_tabular_df = train_tabular_df[np.isfinite(train_tabular_df["dtvt"])].reset_index(drop=True)

feature_columns = []
for col in train_tabular_df.columns:
    if col in BASE_IGNORE_COLUMNS:
        continue
    if col == "TVT" and col in train_tabular_df.columns:
        continue
    if col == "TVT_input" and col in train_tabular_df.columns:
        continue
    if col == "dtvt":
        continue
    if train_tabular_df[col].dtype.kind in "biufc" or col not in ["WELLNAME"]:
        if col != "WELLNAME":
            feature_columns.append(col)

feature_columns = [c for c in feature_columns if c not in {"Typewell_GR", "Typewell_TVT", "tvt_missing_fraction", "tvt_valid_prefix_len"}]
feature_columns = list(dict.fromkeys(feature_columns))
assert "tvt_missing_fraction" not in feature_columns
assert "tvt_valid_prefix_len" not in feature_columns

X_train_tabular = train_tabular_df[feature_columns].apply(pd.to_numeric, errors="coerce")
y_train = train_tabular_df["dtvt"].to_numpy(dtype=np.float32)
groups = train_tabular_df["WELLNAME"].to_numpy()
X_test_tabular = test_tabular_df[feature_columns].apply(pd.to_numeric, errors="coerce")

test_surface_nan_postfill = test_tabular_df[SURFACE_COLUMNS].isna().sum().to_dict()
print(f"Train tabular dataframe shape: {train_tabular_df.shape}")
print(f"Test tabular dataframe shape: {test_tabular_df.shape}")
print(f"Feature matrix shape: {X_train_tabular.shape} | Test matrix shape: {X_test_tabular.shape}")
print(f"Unique train wells: {pd.Series(groups).nunique()} | Unique test wells: {test_tabular_df['WELLNAME'].nunique()}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns[:20]}")
print(f"Post-fill test surface NaN counts: {test_surface_nan_postfill}")


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mse = mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mse))


def _fit_fold_tabular(model_name: str, fold_idx: int, train_idx: np.ndarray, val_idx: np.ndarray, X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame):
    X_tr = X.iloc[train_idx].astype(np.float32)
    y_tr = y[train_idx]
    X_va = X.iloc[val_idx].astype(np.float32)
    y_va = y[val_idx]
    X_te = X_test.astype(np.float32)

    if model_name == "lgbm":
        model = lgb.LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.015,         # Slightly lowered to smooth out the cumsum steps
            num_leaves=63,               # Reduced from 128 to prevent overfitting to local rock wiggles
            min_child_samples=50,        # Forces leaves to represent broader geological trends
            subsample=0.7,               # Increased row-level diversity
            colsample_bytree=0.65,       # Aggressively forces trees to look at different feature subsets
            reg_alpha=0.5,               # Increased L1 regularization to weed out noisy features
            reg_lambda=1.0,              # Increased L2 regularization to stabilize step weights
            extra_trees=True,            # Activates Extremely Randomized Trees for better generalization
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            objective="regression",
            force_col_wise=True,
            verbosity=-1,
        )
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
    elif model_name == "catboost":
        model = CatBoostRegressor(
            iterations=5000,
            learning_rate=0.025,
            depth=6,                     # Reduced from 8 to force broader split decisions
            l2_leaf_reg=6.0,             # Increased from 3.0 to heavily penalize wild trajectory spikes
            random_strength=1.5,         # Adds perturbation to split scores to fight spatial memorization
            subsample=0.7,               # Introduced row sub-sampling for CatBoost
            bootstrap_type="MVS",        # Minimum Variance Sampling for optimal CPU/GPU tree variance
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=SEED,
            od_type="Iter",
            od_wait=150,
            verbose=False,
            thread_count=MODEL_THREAD_COUNT,
            allow_writing_files=False,
            task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    elif model_name == "xgboost":
        model = XGBRegressor(
            n_estimators=5000,
            learning_rate=0.02,
            max_depth=6,                 # Reduced from 8 to mirror the shallower structure of CatBoost
            min_child_weight=10,         # Requires more directional evidence before committing to a split
            subsample=0.7,               # Dropped from 0.8 to increase tree diversity
            colsample_bytree=0.65,       # Dropped from 0.8 to combat spatial coordinate memorization
            reg_alpha=0.5,               # Introduced L1 regularization
            reg_lambda=2.0,              # Increased L2 regularization to steady cumulative variances
            gamma=0.2,                   # Pseudo-pruning: requires a clear structural benefit to split
            early_stopping_rounds=150,
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=MODEL_THREAD_COUNT,
            tree_method="hist",
            verbosity=0,
        )
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    val_pred = model.predict(X_va)
    test_pred = model.predict(X_te)
    fold_rmse = rmse(y_va, val_pred)
    return {
        "fold_idx": fold_idx,
        "model": model,
        "val_idx": val_idx,
        "val_pred": val_pred,
        "test_pred": test_pred,
        "fold_rmse": fold_rmse,
    }


def train_groupkfold_tabular(model_name: str, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, X_test: pd.DataFrame, n_splits: int = N_SPLITS):
    """Parallelized GroupKFold training loop for one tabular model family."""
    gkf = GroupKFold(n_splits=n_splits)
    fold_splits = list(gkf.split(X, y, groups))
    fold_results = Parallel(n_jobs=TABULAR_PARALLEL_JOBS, prefer="processes")(
        delayed(_fit_fold_tabular)(model_name, fold_idx, train_idx, val_idx, X, y, X_test)
        for fold_idx, (train_idx, val_idx) in enumerate(fold_splits)
    )
    oof = np.zeros(len(X), dtype=np.float32)
    test_pred = np.zeros(len(X_test), dtype=np.float32)
    models = []
    for res in sorted(fold_results, key=lambda d: d["fold_idx"]):
        oof[res["val_idx"]] = res["val_pred"]
        test_pred += res["test_pred"] / n_splits
        models.append(res["model"])
    score = rmse(y, oof)
    return {
        "model_name": model_name,
        "oof": oof,
        "test_pred": test_pred,
        "score": score,
        "models": models,
    }


tabular_results = {}
for family_name in ["lgbm", "catboost", "xgboost"]:
    result = train_groupkfold_tabular(family_name, X_train_tabular, y_train, groups, X_test_tabular, N_SPLITS)
    tabular_results[family_name] = result
    print(f"{family_name.upper()} OOF dtvt RMSE: {result['score']:.6f}")

print("Tabular OOF shapes:")
for family_name, result in tabular_results.items():
    print(f"  {family_name}: {result['oof'].shape} | test: {result['test_pred'].shape}")


In [ ]:
def _fill_series_for_alignment(values: np.ndarray) -> np.ndarray:
    """Interpolate missing values before alignment calculations."""
    series = pd.Series(values.astype(float))
    series = series.interpolate(limit_direction="both").bfill().ffill()
    return series.to_numpy(dtype=np.float32)


def normalized_cross_correlation_lag(horizontal_gr: np.ndarray, vertical_gr: np.ndarray, max_lag: int | None = None) -> int:
    """Estimate a global lag using normalized cross-correlation over resampled GR sequences."""
    h = _fill_series_for_alignment(horizontal_gr)
    v = _fill_series_for_alignment(vertical_gr)
    common_len = int(min(max(len(h), len(v)), 512))
    x_common = np.linspace(0.0, 1.0, common_len)
    h_resampled = np.interp(x_common, np.linspace(0.0, 1.0, len(h)), (h - h.mean()) / (h.std() + 1e-6))
    v_resampled = np.interp(x_common, np.linspace(0.0, 1.0, len(v)), (v - v.mean()) / (v.std() + 1e-6))
    if max_lag is None:
        max_lag = max(1, min(common_len // 4, 75))
    best_lag = 0
    best_score = -np.inf
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            h_slice = h_resampled[-lag:]
            v_slice = v_resampled[: len(h_slice)]
        elif lag > 0:
            h_slice = h_resampled[: common_len - lag]
            v_slice = v_resampled[lag: lag + len(h_slice)]
        else:
            h_slice = h_resampled
            v_slice = v_resampled
        if len(h_slice) < 3:
            continue
        score = float(np.dot(h_slice, v_slice) / (np.linalg.norm(h_slice) * np.linalg.norm(v_slice) + 1e-6))
        if score > best_score:
            best_score = score
            best_lag = lag
    return int(best_lag)


def classical_alignment_predict(horizontal_df: pd.DataFrame, typewell_df: pd.DataFrame) -> np.ndarray:
    """Generate a non-neural dtvt baseline using GR correlation and shifted TVT interpolation."""
    h_gr = pd.to_numeric(horizontal_df.get("GR"), errors="coerce").to_numpy(dtype=float)
    v_gr = pd.to_numeric(typewell_df.get("GR"), errors="coerce").to_numpy(dtype=float)
    v_tvt = pd.to_numeric(typewell_df.get("TVT"), errors="coerce").to_numpy(dtype=float)
    h_len = len(h_gr)
    v_len = len(v_tvt)
    lag = normalized_cross_correlation_lag(h_gr, v_gr)
    v_positions = np.arange(v_len, dtype=float) + lag * (max(h_len, v_len) - 1) / max(1, min(max(h_len, v_len), 512) - 1)
    v_positions = np.clip(v_positions, 0.0, max(v_len - 1, 0))
    target_positions = np.linspace(0.0, max(v_len - 1, 0), h_len)
    aligned_tvt = np.interp(target_positions, np.arange(v_len, dtype=float), _fill_series_for_alignment(v_tvt))
    aligned_dtvt = np.diff(aligned_tvt, prepend=aligned_tvt[:1]).astype(np.float32)
    if len(aligned_dtvt):
        aligned_dtvt[0] = 0.0
    return aligned_dtvt


classical_oof = np.zeros(len(train_tabular_df), dtype=np.float32)
fold_scores = []
unique_train_wells = sorted(train_well_map)
gkf = GroupKFold(n_splits=N_SPLITS)
well_groups = np.array(unique_train_wells)
for fold_idx, (_, val_well_idx) in enumerate(gkf.split(well_groups, well_groups, well_groups)):
    fold_wells = well_groups[val_well_idx]
    fold_rows = []
    fold_true = []
    fold_pred = []
    for well_name in fold_wells:
        bundle = train_well_map[well_name]
        pred = classical_alignment_predict(bundle["horizontal"], bundle["typewell"])
        mask = train_tabular_df["WELLNAME"].eq(well_name).to_numpy()
        classical_oof[mask] = pred[: mask.sum()]
        fold_rows.append(mask.sum())
        fold_true.append(train_tabular_df.loc[mask, "dtvt"].to_numpy())
        fold_pred.append(pred[: mask.sum()])
    fold_true_arr = np.concatenate(fold_true)
    fold_pred_arr = np.concatenate(fold_pred)
    fold_rmse = rmse(fold_true_arr, fold_pred_arr)
    fold_scores.append(fold_rmse)
    print(f"Classical fold {fold_idx + 1}/{N_SPLITS} dtvt RMSE: {fold_rmse:.6f} | wells: {len(fold_wells)} | rows: {int(np.sum(fold_rows))}")

classical_rmse = rmse(y_train, classical_oof)
print(f"Classical alignment baseline OOF dtvt RMSE: {classical_rmse:.6f}")
print(f"Classical OOF shape: {classical_oof.shape}")


In [ ]:
def _safe_standardize(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    mean = float(np.nanmean(values)) if np.isfinite(values).any() else 0.0
    std = float(np.nanstd(values)) if np.isfinite(values).any() else 1.0
    std = std if std > 1e-6 else 1.0
    filled = pd.Series(values).interpolate(limit_direction="both").bfill().ffill().to_numpy(dtype=np.float32)
    return (filled - mean) / std


def _resample_segment(values: np.ndarray, start_frac: float, end_frac: float, target_len: int) -> np.ndarray:
    filled = pd.Series(np.asarray(values, dtype=np.float32)).interpolate(limit_direction="both").bfill().ffill().to_numpy(dtype=np.float32)
    if len(filled) == 0 or target_len <= 0:
        return np.full(target_len, np.nan, dtype=np.float32)
    source_positions = np.linspace(0.0, 1.0, len(filled), dtype=np.float32)
    target_positions = np.linspace(start_frac, end_frac, target_len, dtype=np.float32)
    return np.interp(target_positions, source_positions, filled).astype(np.float32)


class WellSequenceDataset(torch.utils.data.Dataset):
    """Chunked per-well dataset exposing bounded GR windows and dtvt targets."""

    def __init__(self, wells: List[dict], include_target: bool = True, history_window: int = 256, forecast_horizon: int = 1024):
        self.wells = wells
        self.include_target = include_target
        self.history_window = history_window
        self.forecast_horizon = forecast_horizon
        self.window_size = history_window + forecast_horizon
        self.stride = history_window
        self.samples = []
        for bundle in wells:
            h = bundle["horizontal"].reset_index(drop=True)
            v = bundle["typewell"].reset_index(drop=True)
            total_len = len(h)
            if total_len == 0:
                continue
            start_positions = list(range(0, total_len, self.stride))
            if not start_positions:
                start_positions = [0]
            for start in start_positions:
                end = min(total_len, start + self.window_size)
                if end - start < 4:
                    continue
                self.samples.append({"bundle": bundle, "start": start, "end": end})

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample_info = self.samples[idx]
        bundle = sample_info["bundle"]
        h = bundle["horizontal"].reset_index(drop=True)
        v = bundle["typewell"].reset_index(drop=True)
        start = int(sample_info["start"])
        end = int(sample_info["end"])
        window_len = end - start
        total_len = max(len(h), 1)
        start_frac = start / max(total_len - 1, 1)
        end_frac = (end - 1) / max(total_len - 1, 1)

        h_gr = pd.to_numeric(h.get("GR"), errors="coerce").to_numpy(dtype=np.float32)
        v_gr = pd.to_numeric(v.get("GR"), errors="coerce").to_numpy(dtype=np.float32)
        h_tvt = pd.to_numeric(h.get("TVT"), errors="coerce").diff().fillna(0.0).to_numpy(dtype=np.float32) if "TVT" in h.columns else np.full(len(h_gr), np.nan, dtype=np.float32)

        h_slice = h_gr[start:end]
        h_pos = np.linspace(0.0, 1.0, window_len, dtype=np.float32)
        v_slice = _resample_segment(v_gr, start_frac, end_frac, window_len)
        v_pos = np.linspace(start_frac, end_frac, window_len, dtype=np.float32)
        h_feat = np.stack([_safe_standardize(h_slice), h_pos], axis=-1)
        v_feat = np.stack([_safe_standardize(v_slice), v_pos], axis=-1)
        row_idx = h["row_idx"].to_numpy(dtype=np.int64)[start:end]

        sample = {
            "wellname": bundle["WELLNAME"],
            "horizontal_features": torch.tensor(h_feat, dtype=torch.float32),
            "vertical_features": torch.tensor(v_feat, dtype=torch.float32),
            "horizontal_mask": torch.ones(window_len, dtype=torch.bool),
            "vertical_mask": torch.ones(window_len, dtype=torch.bool),
            "row_idx": torch.tensor(row_idx, dtype=torch.long),
            "chunk_start": start,
            "chunk_end": end,
        }
        if self.include_target:
            sample["target"] = torch.tensor(h_tvt[start:end], dtype=torch.float32)
        else:
            sample["target"] = torch.full((window_len,), float("nan"), dtype=torch.float32)
        return sample


def sequence_collate(batch: List[dict]) -> dict:
    """Pad variable-length chunks into a dense batch."""
    h_feats = [item["horizontal_features"] for item in batch]
    v_feats = [item["vertical_features"] for item in batch]
    targets = [item["target"] for item in batch]
    row_idx = [item["row_idx"] for item in batch]
    wellnames = [item["wellname"] for item in batch]
    h_mask = [item["horizontal_mask"] for item in batch]
    v_mask = [item["vertical_mask"] for item in batch]
    batch_dict = {
        "horizontal_features": torch.nn.utils.rnn.pad_sequence(h_feats, batch_first=True, padding_value=0.0),
        "vertical_features": torch.nn.utils.rnn.pad_sequence(v_feats, batch_first=True, padding_value=0.0),
        "target": torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=float("nan")),
        "row_idx": torch.nn.utils.rnn.pad_sequence(row_idx, batch_first=True, padding_value=-1),
        "wellname": wellnames,
        "horizontal_mask": torch.nn.utils.rnn.pad_sequence(h_mask, batch_first=True, padding_value=0),
        "vertical_mask": torch.nn.utils.rnn.pad_sequence(v_mask, batch_first=True, padding_value=0),
    }
    return batch_dict


class ConvEncoder(nn.Module):
    """Dual 1D CNN encoder for local stratigraphic signatures."""

    def __init__(self, input_dim: int = 2, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.block1 = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.out_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.transpose(1, 2)
        x = self.block1(x)
        x = self.block2(x)
        x = x.transpose(1, 2)
        return self.out_norm(x)


class CrossAttentionBlock(nn.Module):
    """Cross-attention with horizontal queries and vertical keys/values."""

    def __init__(self, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = float(hidden_dim) ** -0.5

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, q_mask: torch.Tensor | None = None, kv_mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        q = self.q_proj(q)
        k = self.k_proj(k)
        v = self.v_proj(v)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if kv_mask is not None:
            min_val = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~kv_mask.unsqueeze(1), min_val)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = self.out_proj(context)
        if q_mask is not None:
            context = context * q_mask.unsqueeze(-1).to(context.dtype)
        return context, attn


class SequenceAlignmentNet(nn.Module):
    """End-to-end alignment network that predicts dtvt for every bounded horizontal chunk."""

    def __init__(self, input_dim: int = 2, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.horizontal_encoder = ConvEncoder(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout)
        self.vertical_encoder = ConvEncoder(input_dim=input_dim, hidden_dim=hidden_dim, dropout=dropout)
        self.cross_attention = CrossAttentionBlock(hidden_dim=hidden_dim, dropout=dropout)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, horizontal_features: torch.Tensor, vertical_features: torch.Tensor, horizontal_mask: torch.Tensor | None = None, vertical_mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.horizontal_encoder(horizontal_features)
        v = self.vertical_encoder(vertical_features)
        context, attn = self.cross_attention(h, v, v, q_mask=horizontal_mask, kv_mask=vertical_mask)
        fusion = torch.cat([h, context, h - context, h * context], dim=-1)
        pred = self.regressor(fusion).squeeze(-1)
        return pred, attn


def masked_regression_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    valid = mask & torch.isfinite(target)
    if not valid.any():
        return pred.sum() * 0.0
    pred_valid = pred[valid]
    target_valid = target[valid]
    mse = F.mse_loss(pred_valid, target_valid)
    huber = F.smooth_l1_loss(pred_valid, target_valid, beta=1.0)
    return 0.5 * mse + 0.5 * huber


def monotonicity_penalty(pred: torch.Tensor, attn: torch.Tensor, mask: torch.Tensor, jump_threshold: float = 5.0) -> torch.Tensor:
    penalties = []
    if pred.size(1) > 1:
        d1 = pred[:, 1:] - pred[:, :-1]
        valid1 = mask[:, 1:] & mask[:, :-1]
        penalties.append((F.relu(d1.abs() - jump_threshold).pow(2) * valid1).sum() / valid1.sum().clamp_min(1))
        if pred.size(1) > 2:
            d2 = d1[:, 1:] - d1[:, :-1]
            valid2 = mask[:, 2:] & mask[:, 1:-1] & mask[:, :-2]
            penalties.append((d2.pow(2) * valid2).sum() / valid2.sum().clamp_min(1))
    if attn is not None:
        vertical_index = torch.arange(attn.size(-1), device=attn.device, dtype=attn.dtype)
        expected_vertical_index = torch.matmul(attn, vertical_index)
        if expected_vertical_index.size(1) > 1:
            idx_d1 = expected_vertical_index[:, 1:] - expected_vertical_index[:, :-1]
            valid1 = mask[:, 1:] & mask[:, :-1]
            penalties.append((F.relu(-idx_d1).pow(2) * valid1).sum() / valid1.sum().clamp_min(1))
            if expected_vertical_index.size(1) > 2:
                idx_d2 = idx_d1[:, 1:] - idx_d1[:, :-1]
                valid2 = mask[:, 2:] & mask[:, 1:-1] & mask[:, :-2]
                penalties.append((idx_d2.pow(2) * valid2).sum() / valid2.sum().clamp_min(1))
    return sum(penalties) if penalties else pred.new_tensor(0.0)


def _aggregate_sequence_predictions(per_sample_predictions: Dict[str, list[tuple[np.ndarray, np.ndarray]]], well_lengths: Dict[str, int]) -> Dict[str, np.ndarray]:
    aggregated = {}
    for wellname, records in per_sample_predictions.items():
        length = int(well_lengths.get(wellname, 0))
        values = np.full(length, np.nan, dtype=np.float32)
        counts = np.zeros(length, dtype=np.float32)
        for rows, pred in records:
            valid = (rows >= 0) & (rows < length)
            rows = rows[valid].astype(int)
            pred = pred[valid].astype(np.float32)
            values[rows] = np.nan_to_num(values[rows], nan=0.0) + pred
            counts[rows] += 1.0
        mask = counts > 0
        if mask.any():
            values[mask] = values[mask] / counts[mask]
        aggregated[wellname] = values
    return aggregated


def sequence_batch_rmse(model: nn.Module, loader: torch.utils.data.DataLoader, well_lengths: Dict[str, int] | None = None) -> Dict[str, np.ndarray]:
    model.eval()
    per_sample_predictions: Dict[str, list[tuple[np.ndarray, np.ndarray]]] = {}
    with torch.no_grad():
        for batch in loader:
            horizontal = batch["horizontal_features"].to(DEVICE)
            vertical = batch["vertical_features"].to(DEVICE)
            h_mask = batch["horizontal_mask"].to(DEVICE).bool()
            v_mask = batch["vertical_mask"].to(DEVICE).bool()
            pred, attn = model(horizontal, vertical, h_mask, v_mask)
            pred = pred.detach().cpu().numpy()
            row_idx = batch["row_idx"].cpu().numpy()
            for i, wellname in enumerate(batch["wellname"]):
                length = int(h_mask[i].sum().item())
                per_sample_predictions.setdefault(wellname, []).append((row_idx[i, :length].copy(), pred[i, :length].copy()))
    if well_lengths is None:
        well_lengths = {}
        for wellname, records in per_sample_predictions.items():
            max_idx = max(int(np.nanmax(rows)) for rows, _ in records if len(rows)) if records else -1
            well_lengths[wellname] = max_idx + 1 if max_idx >= 0 else 0
    return _aggregate_sequence_predictions(per_sample_predictions, well_lengths)


def train_sequence_fold(train_wells_fold: List[dict], val_wells_fold: List[dict], epochs: int = MAX_SEQ_EPOCHS, batch_size: int = 8) -> Tuple[SequenceAlignmentNet, float, Dict[str, np.ndarray]]:
    train_loader = torch.utils.data.DataLoader(WellSequenceDataset(train_wells_fold, include_target=True), batch_size=batch_size, shuffle=True, collate_fn=sequence_collate)
    val_loader = torch.utils.data.DataLoader(WellSequenceDataset(val_wells_fold, include_target=True), batch_size=batch_size, shuffle=False, collate_fn=sequence_collate)

    sample_batch = next(iter(train_loader))
    print(f"Sample sequence batch shapes -> H: {tuple(sample_batch['horizontal_features'].shape)}, V: {tuple(sample_batch['vertical_features'].shape)}, y: {tuple(sample_batch['target'].shape)}")

    model = SequenceAlignmentNet(input_dim=2, hidden_dim=64, dropout=0.1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_CUDA)

    best_state = None
    best_rmse = float("inf")
    stale_epochs = 0
    val_well_lengths = {bundle["WELLNAME"]: len(bundle["horizontal"]) for bundle in val_wells_fold}
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            horizontal = batch["horizontal_features"].to(DEVICE)
            vertical = batch["vertical_features"].to(DEVICE)
            target = batch["target"].to(DEVICE)
            h_mask = batch["horizontal_mask"].to(DEVICE).bool()
            v_mask = batch["vertical_mask"].to(DEVICE).bool()
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_CUDA):
                pred, attn = model(horizontal, vertical, h_mask, v_mask)
                base_loss = masked_regression_loss(pred, target, h_mask)
                smooth_penalty = monotonicity_penalty(pred, attn, h_mask)
                loss = base_loss + 0.1 * smooth_penalty
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        val_prediction_map = sequence_batch_rmse(model, val_loader, val_well_lengths)
        val_true = []
        val_pred = []
        for bundle in val_wells_fold:
            wellname = bundle["WELLNAME"]
            true_dtvt = pd.to_numeric(bundle["horizontal"].get("TVT"), errors="coerce").diff().fillna(0.0).to_numpy(dtype=np.float32)
            pred_dtvt = val_prediction_map.get(wellname, np.full(len(true_dtvt), np.nan, dtype=np.float32))[: len(true_dtvt)]
            valid = np.isfinite(true_dtvt) & np.isfinite(pred_dtvt)
            if valid.any():
                val_true.append(true_dtvt[valid])
                val_pred.append(pred_dtvt[valid])
        val_true_arr = np.concatenate(val_true) if val_true else np.array([], dtype=np.float32)
        val_pred_arr = np.concatenate(val_pred) if val_pred else np.array([], dtype=np.float32)
        epoch_rmse = rmse(val_true_arr, val_pred_arr) if len(val_true_arr) else float("inf")
        scheduler.step(epoch_rmse)
        print(f"Sequence epoch {epoch + 1:02d}/{epochs} | train loss {np.mean(train_losses):.6f} | val dtvt RMSE {epoch_rmse:.6f}")
        if epoch_rmse < best_rmse - 1e-6:
            best_rmse = epoch_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= SEQ_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    val_predictions = sequence_batch_rmse(model, val_loader, val_well_lengths)
    return model, best_rmse, val_predictions

## Sequence training
# sequence_oof = np.full(len(train_tabular_df), np.nan, dtype=np.float32)
# sequence_test_preds_per_fold = []
# sequence_models = []
# sequence_fold_scores = []
# train_well_names_order = np.array(sorted(train_well_map))
# well_group_kfold = GroupKFold(n_splits=N_SPLITS)
# for fold_idx, (train_well_idx, val_well_idx) in enumerate(well_group_kfold.split(train_well_names_order, train_well_names_order, train_well_names_order)):
#     train_fold_names = train_well_names_order[train_well_idx].tolist()
#     val_fold_names = train_well_names_order[val_well_idx].tolist()
#     train_fold_wells = [train_well_map[name] for name in train_fold_names]
#     val_fold_wells = [train_well_map[name] for name in val_fold_names]
#     print(f"\nSequence fold {fold_idx + 1}/{N_SPLITS} | train wells: {len(train_fold_wells)} | val wells: {len(val_fold_wells)}")
#     model, fold_rmse, val_predictions = train_sequence_fold(train_fold_wells, val_fold_wells, epochs=MAX_SEQ_EPOCHS, batch_size=2)
#     sequence_models.append(model)
#     sequence_fold_scores.append(fold_rmse)
#     for well_name, pred in val_predictions.items():
#         mask = train_tabular_df["WELLNAME"].eq(well_name).to_numpy()
#         sequence_oof[mask] = pred[: mask.sum()]
#     test_loader = torch.utils.data.DataLoader(WellSequenceDataset(test_wells, include_target=False), batch_size=2, shuffle=False, collate_fn=sequence_collate)
#     test_well_lengths = {bundle["WELLNAME"]: len(bundle["horizontal"]) for bundle in test_wells}
#     fold_test_predictions = sequence_batch_rmse(model, test_loader, test_well_lengths)
#     sequence_test_preds_per_fold.append(fold_test_predictions)
#     print(f"Sequence fold {fold_idx + 1} validation dtvt RMSE: {fold_rmse:.6f}")

# sequence_oof = np.nan_to_num(sequence_oof, nan=classical_oof)
# sequence_rmse = rmse(y_train, sequence_oof)
# print(f"Sequence alignment OOF dtvt RMSE: {sequence_rmse:.6f}")
# print(f"Sequence OOF shape: {sequence_oof.shape}")

# sample_batch = next(iter(torch.utils.data.DataLoader(WellSequenceDataset(train_wells[:2], include_target=True), batch_size=2, shuffle=False, collate_fn=sequence_collate)))
# with torch.no_grad():
#     sample_pred, sample_attn = sequence_models[0](sample_batch["horizontal_features"].to(DEVICE), sample_batch["vertical_features"].to(DEVICE), sample_batch["horizontal_mask"].to(DEVICE).bool(), sample_batch["vertical_mask"].to(DEVICE).bool())
# print(f"Sample model output shapes -> pred: {tuple(sample_pred.shape)}, attn: {tuple(sample_attn.shape)}")


In [ ]:
def _stack_oof_predictions(*arrays: np.ndarray) -> np.ndarray:
    return np.column_stack([np.asarray(a, dtype=np.float32) for a in arrays])


def fit_oof_ridge(meta_features: np.ndarray, y: np.ndarray, groups: np.ndarray, n_splits: int = N_SPLITS, alpha: float = 1.0):
    """Train an OOF Ridge meta-learner with GroupKFold."""
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y), dtype=np.float32)
    fold_models = []
    fold_rmses = []
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(meta_features, y, groups)):
        model = Ridge(alpha=alpha, random_state=SEED)
        model.fit(meta_features[train_idx], y[train_idx])
        val_pred = model.predict(meta_features[val_idx])
        oof[val_idx] = val_pred
        fold_models.append(model)
        fold_rmses.append(rmse(y[val_idx], val_pred))
        print(f"Ridge meta fold {fold_idx + 1}/{n_splits} RMSE: {fold_rmses[-1]:.6f}")
    overall_rmse = rmse(y, oof)
    return oof, fold_models, overall_rmse


tabular_oof_stack = _stack_oof_predictions(
    tabular_results["lgbm"]["oof"],
    tabular_results["catboost"]["oof"],
    tabular_results["xgboost"]["oof"],
)
full_oof_stack = _stack_oof_predictions(
    tabular_results["lgbm"]["oof"],
    tabular_results["catboost"]["oof"],
    tabular_results["xgboost"]["oof"],
    # sequence_oof,
)

tabular_meta_oof, tabular_meta_models, tabular_meta_rmse = fit_oof_ridge(tabular_oof_stack, y_train, groups, n_splits=N_SPLITS, alpha=1.0)
full_meta_oof, full_meta_models, full_meta_rmse = fit_oof_ridge(full_oof_stack, y_train, groups, n_splits=N_SPLITS, alpha=1.0)

# sequence_gate_rmse = sequence_rmse
# use_full_fusion = bool(full_meta_rmse < tabular_meta_rmse and sequence_gate_rmse <= tabular_meta_rmse * 1.05)
use_full_fusion = 0
blend_mode = "full_fusion" if use_full_fusion else "tabular_only"
final_oof = full_meta_oof if use_full_fusion else tabular_meta_oof
final_meta_features_train = full_oof_stack if use_full_fusion else tabular_oof_stack
final_meta_model = Ridge(alpha=1.0, random_state=SEED).fit(final_meta_features_train, y_train)
final_train_rmse = rmse(y_train, final_oof)

print(f"Tabular-only Ridge OOF dtvt RMSE: {tabular_meta_rmse:.6f}")
print(f"Full fusion Ridge OOF dtvt RMSE: {full_meta_rmse:.6f}")
# print(f"Sequence gate dtvt RMSE: {sequence_gate_rmse:.6f}")
print(f"Selected blend mode: {blend_mode}")
print(f"Final blended OOF dtvt RMSE: {final_train_rmse:.6f}")
print(f"Meta feature shapes -> tabular: {tabular_oof_stack.shape}, full: {full_oof_stack.shape}")


def compute_error_decomposition(df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Create per-well diagnostic metrics for residual analysis on dtvt."""
    rows = []
    for well_name, well_df in df.groupby("WELLNAME", sort=False):
        idx = well_df.index.to_numpy()
        actual = y_true[idx]
        pred = y_pred[idx]
        err = pred - actual
        rows.append(
            {
                "WELLNAME": well_name,
                "n_rows": int(len(idx)),
                "rmse": rmse(actual, pred),
                "mae": float(mean_absolute_error(actual, pred)),
                "directional_bias": float(err.mean()),
                "spatial_drift": float(np.abs(np.cumsum(err)).mean()),
            }
        )
    return pd.DataFrame(rows).sort_values("rmse", ascending=False).reset_index(drop=True)


error_report_df = compute_error_decomposition(train_tabular_df, y_train, final_oof)
print(f"Error decomposition report shape: {error_report_df.shape}")
print(error_report_df.head())


def plot_well_alignment(wellname: str, md: np.ndarray, actual: np.ndarray, predicted: np.ndarray, max_points: int = 400) -> None:
    """Plot predicted vs actual dtvt along measured depth for one well."""
    plt.figure(figsize=(12, 4))
    if len(md) > max_points:
        stride = max(1, len(md) // max_points)
        md = md[::stride]
        actual = actual[::stride]
        predicted = predicted[::stride]
    plt.plot(md, actual, label="Actual dtvt", linewidth=2)
    plt.plot(md, predicted, label="Predicted dtvt", linewidth=2)
    plt.title(f"{wellname} - Predicted dtvt vs Actual dtvt")
    plt.xlabel("Measured Depth (MD)")
    plt.ylabel("dtvt")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


example_well = error_report_df.iloc[0]["WELLNAME"] if len(error_report_df) else train_tabular_df["WELLNAME"].iloc[0]
example_mask = train_tabular_df["WELLNAME"].eq(example_well).to_numpy()
example_md = pd.to_numeric(train_tabular_df.loc[example_mask, "MD"], errors="coerce").to_numpy(dtype=float)
example_actual = y_train[example_mask]
example_pred = final_oof[example_mask]
print(f"Plot diagnostic example: {example_well} | rows: {example_mask.sum()}")
plot_well_alignment(example_well, example_md, example_actual, example_pred)


In [ ]:
def _average_fold_test_predictions(per_fold_predictions: List[Dict[str, np.ndarray]], well_names: List[str]) -> Dict[str, np.ndarray]:
    """Average per-fold well predictions into a single test prediction per well."""
    well_to_predictions = {name: [] for name in well_names}
    for fold_pred in per_fold_predictions:
        for name in well_names:
            if name in fold_pred:
                well_to_predictions[name].append(fold_pred[name])
    averaged = {}
    for name, preds in well_to_predictions.items():
        if preds:
            stacked = np.vstack([np.asarray(p, dtype=np.float32) for p in preds])
            averaged[name] = np.nanmean(stacked, axis=0).astype(np.float32)
        else:
            averaged[name] = np.full(len(test_well_map[name]["horizontal"]), np.nan, dtype=np.float32)
    return averaged


def build_prediction_frame_from_well_map(well_map: Dict[str, dict], per_well_predictions: Dict[str, np.ndarray], include_eval_only: bool = True) -> pd.DataFrame:
    """Materialize row-level predictions and id strings in submission format."""
    rows = []
    for well_name, bundle in well_map.items():
        horizontal = bundle["horizontal"]
        boundary = bundle["boundary"]
        pred = np.asarray(per_well_predictions[well_name], dtype=np.float32)
        if len(pred) < len(horizontal):
            pad_value = pred[-1] if len(pred) else 0.0
            pred = np.pad(pred, (0, len(horizontal) - len(pred)), constant_values=pad_value)
        pred = pred[: len(horizontal)]
        if include_eval_only:
            mask = horizontal["row_idx"].to_numpy() >= int(boundary["tvt_eval_start_idx"])
        else:
            mask = np.ones(len(horizontal), dtype=bool)
        sel = horizontal.loc[mask, ["WELLNAME", "row_idx"]].copy()
        sel["id"] = sel["WELLNAME"].astype(str) + "_" + sel["row_idx"].astype(int).astype(str)
        sel["pred"] = np.nan_to_num(pred[mask], nan=0.0, posinf=0.0, neginf=0.0)
        rows.append(sel[["id", "pred"]])
    return pd.concat(rows, ignore_index=True)


def reconstruct_absolute_tvt(bundle: dict, dtvt_predictions: np.ndarray) -> np.ndarray:
    """Rebuild absolute TVT from dtvt predictions using the last valid TVT_input anchor."""
    horizontal = bundle["horizontal"].reset_index(drop=True)
    boundary = bundle["boundary"]
    anchor_idx = int(boundary["tvt_last_valid_idx"])
    anchor_value = float(boundary["tvt_last_valid_value"])
    absolute = pd.to_numeric(horizontal.get("TVT_input"), errors="coerce").to_numpy(dtype=np.float32) if "TVT_input" in horizontal.columns else np.full(len(horizontal), np.nan, dtype=np.float32)
    dtvt_predictions = np.asarray(dtvt_predictions, dtype=np.float32)
    if len(dtvt_predictions) < len(horizontal):
        pad_value = dtvt_predictions[-1] if len(dtvt_predictions) else 0.0
        dtvt_predictions = np.pad(dtvt_predictions, (0, len(horizontal) - len(dtvt_predictions)), constant_values=pad_value)
    dtvt_predictions = dtvt_predictions[: len(horizontal)]
    if anchor_idx >= 0:
        absolute[anchor_idx] = anchor_value
        if anchor_idx + 1 < len(horizontal):
            future_dtvt = np.nan_to_num(dtvt_predictions[anchor_idx + 1:], nan=0.0, posinf=0.0, neginf=0.0)
            absolute[anchor_idx + 1:] = anchor_value + np.cumsum(future_dtvt)
    return absolute


# Base test predictions for tabular models come from fold-averaged predictions already computed in cell 3.
tabular_test_stack = _stack_oof_predictions(
    tabular_results["lgbm"]["test_pred"],
    tabular_results["catboost"]["test_pred"],
    tabular_results["xgboost"]["test_pred"],
)

# sequence_test_avg = _average_fold_test_predictions(sequence_test_preds_per_fold, list(test_well_map.keys()))
test_surface_lookup = {name: bundle.get("surface_lookup", build_surface_lookup_from_typewell(bundle["typewell"])) for name, bundle in test_well_map.items()}
validated_test_well_map = {}
for name, bundle in test_well_map.items():
    updated_bundle = dict(bundle)
    updated_bundle["surface_lookup"] = test_surface_lookup[name]
    updated_bundle["horizontal"] = attach_surface_depths(bundle["horizontal"], test_surface_lookup[name])
    validated_test_well_map[name] = updated_bundle

print(f"Deterministic surface lookup prepared for {len(validated_test_well_map)} test wells")
if validated_test_well_map:
    sample_name = next(iter(validated_test_well_map))
    print(f"Sample test surface lookup ({sample_name}): {validated_test_well_map[sample_name]['surface_lookup']}")

# sequence_test_series = build_prediction_frame_from_well_map(validated_test_well_map, sequence_test_avg, include_eval_only=False)
# sequence_test_vector = np.concatenate([
#     sequence_test_avg[name]
#     for name in test_tabular_df["WELLNAME"].groupby(test_tabular_df["WELLNAME"]).groups.keys()
# ]).astype(np.float32)

if use_full_fusion:
    test_meta_features = _stack_oof_predictions(tabular_results["lgbm"]["test_pred"], tabular_results["catboost"]["test_pred"], tabular_results["xgboost"]["test_pred"], sequence_test_vector)
else:
    test_meta_features = tabular_test_stack

final_test_dtvt_pred_all_rows = final_meta_model.predict(test_meta_features).astype(np.float32)
final_test_dtvt_pred_all_rows = np.nan_to_num(final_test_dtvt_pred_all_rows, nan=float(np.nanmean(y_train)), posinf=float(np.nanmax(y_train)), neginf=float(np.nanmin(y_train)))

test_tabular_df = test_tabular_df.reset_index(drop=True).copy()
test_tabular_df["dtvt_prediction"] = final_test_dtvt_pred_all_rows

submission_rows = []
for well_name, bundle in validated_test_well_map.items():
    horizontal = bundle["horizontal"]
    boundary = bundle["boundary"]
    mask = horizontal["row_idx"].to_numpy() >= int(boundary["tvt_eval_start_idx"])
    well_rows = horizontal.loc[mask, ["WELLNAME", "row_idx"]].copy()
    well_dtvt_pred = test_tabular_df.loc[test_tabular_df["WELLNAME"].eq(well_name), "dtvt_prediction"].to_numpy(dtype=np.float32)
    absolute_tvt_pred = reconstruct_absolute_tvt(bundle, well_dtvt_pred)
    well_rows["id"] = well_rows["WELLNAME"].astype(str) + "_" + well_rows["row_idx"].astype(int).astype(str)
    well_rows["tvt"] = np.nan_to_num(absolute_tvt_pred[mask], nan=float(np.nanmean(y_train)), posinf=float(np.nanmax(y_train)), neginf=float(np.nanmin(y_train)))
    submission_rows.append(well_rows[["id", "tvt"]])

submission_df = pd.concat(submission_rows, ignore_index=True)
submission_df = submission_df.dropna(subset=["id", "tvt"]).copy()
submission_df["tvt"] = submission_df["tvt"].astype(float)
submission_df.to_csv("submission.csv", index=False)

print(f"Submission shape: {submission_df.shape}")
print(submission_df.head())
print(f"Final blend mode used for test predictions: {blend_mode}")
print("Submission saved to: submission.csv")
